In [12]:
import pandas as pd
import glob
import os

# Set the directory path
data_dir = "/home/vinh/HS Code/HS_CODE/data/QA data/"

# Find all CSV files that match the pattern QAdataset_1000_*.csv
file_pattern = os.path.join(data_dir, "QAdataset_1000_*.csv")
csv_files = glob.glob(file_pattern)

# Sort files to ensure consistent order
csv_files.sort()

print(f"Found {len(csv_files)} files to merge:")
for file in csv_files:
    print(f"- {os.path.basename(file)}")

# Read and merge all CSV files
dataframes = []
for file in csv_files:
    df = pd.read_csv(file)
    print(f"Loaded {file}: {len(df)} rows")
    dataframes.append(df)

# Concatenate all dataframes
merged_df = pd.concat(dataframes, ignore_index=True)

print(f"\nMerged dataset shape: {merged_df.shape}")
print(f"Total rows: {len(merged_df)}")

# Save the merged dataset
output_file = os.path.join(data_dir, "QAdataset.csv")
merged_df.to_csv(output_file, index=False)

print(f"\nMerged data saved to: {output_file}")

# Display first few rows of merged data
print("\nFirst 5 rows of merged dataset:")
print(merged_df.head())

Found 3 files to merge:
- QAdataset_1000_1.csv
- QAdataset_1000_2.csv
- QAdataset_1000_3.csv
Loaded /home/vinh/HS Code/HS_CODE/data/QA data/QAdataset_1000_1.csv: 3000 rows
Loaded /home/vinh/HS Code/HS_CODE/data/QA data/QAdataset_1000_2.csv: 3000 rows
Loaded /home/vinh/HS Code/HS_CODE/data/QA data/QAdataset_1000_3.csv: 24432 rows

Merged dataset shape: (30432, 3)
Total rows: 30432

Merged data saved to: /home/vinh/HS Code/HS_CODE/data/QA data/QAdataset.csv

First 5 rows of merged dataset:
                       mahs  \
0              ['01012100']   
1  ['01012100', '01012900']   
2              ['01012100']   
3              ['01012900']   
4  ['01012900', '01019000']   

                                              prompt  \
0  Tôi nhập ngựa thuần chủng để nhân giống thì dù...   
1  Ngựa thuần chủng để nhân giống khác gì so với ...   
2  Nếu tôi khai mã 01012900 cho ngựa thuần chủng ...   
3  Tôi nhập ngựa không phải để nhân giống thì dùn...   
4  Nếu nhập ngựa đua, tôi nên dùng mã 01

In [13]:
import os
import json
from sklearn.model_selection import train_test_split

def split_dataset_v2(df, output_dir="/home/vinh/HS Code/HS_CODE/vit5-chatbot-finetune/data"):
    """
    Chia dataset thành train/validation/test với tỷ lệ 7:1.5:1.5 từ dataframe đã có.
    """

    # Đổi tên trường từ prompt/response nếu cần
    processed_data = [
        {"prompt": row["prompt"], "response": row["response"]}
        for _, row in df.iterrows()
    ]

    print(f"Tổng số mẫu dữ liệu: {len(processed_data)}")

    # Chia train/val/test (7/1.5/1.5)
    print("Đang chia dữ liệu...")
    train_data, temp_data = train_test_split(processed_data, test_size=0.3, random_state=42)
    val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

    print(f"Train set: {len(train_data)} mẫu ({len(train_data)/len(processed_data)*100:.1f}%)")
    print(f"Validation set: {len(val_data)} mẫu ({len(val_data)/len(processed_data)*100:.1f}%)")
    print(f"Test set: {len(test_data)} mẫu ({len(test_data)/len(processed_data)*100:.1f}%)")

    os.makedirs(output_dir, exist_ok=True)
    train_path = os.path.join(output_dir, "train_dataset_full.json")
    val_path = os.path.join(output_dir, "val_dataset_full.json")
    test_path = os.path.join(output_dir, "test_dataset_full.json")

    print("Đang lưu dữ liệu...")

    with open(train_path, 'w', encoding='utf-8') as f:
        json.dump(train_data, f, ensure_ascii=False, indent=2)
    print(f"Đã lưu train set tại: {train_path}")

    with open(val_path, 'w', encoding='utf-8') as f:
        json.dump(val_data, f, ensure_ascii=False, indent=2)
    print(f"Đã lưu validation set tại: {val_path}")

    with open(test_path, 'w', encoding='utf-8') as f:
        json.dump(test_data, f, ensure_ascii=False, indent=2)
    print(f"Đã lưu test set tại: {test_path}")

    print("Hoàn thành chia dataset!")

    return {
        'train': train_data,
        'validation': val_data,
        'test': test_data
    }

# Gọi hàm với merged_df
split_dataset_v2(merged_df)

Tổng số mẫu dữ liệu: 30432
Đang chia dữ liệu...
Train set: 21302 mẫu (70.0%)
Validation set: 4565 mẫu (15.0%)
Test set: 4565 mẫu (15.0%)
Đang lưu dữ liệu...
Đã lưu train set tại: /home/vinh/HS Code/HS_CODE/vit5-chatbot-finetune/data/train_dataset_full.json
Đã lưu validation set tại: /home/vinh/HS Code/HS_CODE/vit5-chatbot-finetune/data/val_dataset_full.json
Đã lưu test set tại: /home/vinh/HS Code/HS_CODE/vit5-chatbot-finetune/data/test_dataset_full.json
Hoàn thành chia dataset!


{'train': [{'prompt': 'Máy dệt kim tròn loại 84471100 khác gì so với loại 84471200?',
   'response': 'Sự khác biệt chính là đường kính trục cuốn. 84471100 không quá 165mm, còn 84471200 thì trên 165mm.'},
  {'prompt': 'Tôi nhập giấy cuộn lớn để làm hộp carton, giấy này có phải là thuộc mã 48070000 không?',
   'response': 'Nếu giấy của bạn được tạo thành từ nhiều lớp giấy hoặc bìa dán lại với nhau, chưa tráng hoặc thấm tẩm, thì có thể thuộc mã 48070000.'},
  {'prompt': 'Nếu tôi khai báo nhầm hợp kim fero-niken vào mã khác, có sao không?',
   'response': 'Khai sai mã HS có thể dẫn đến phạt hoặc chậm trễ thông quan. Cần khai đúng mã 72026000.'},
  {'prompt': 'Tôi nhập thép cuộn cán nguội dạng dải, carbon dưới 0.25%, rộng dưới 400mm, mã nào?',
   'response': 'Trường hợp này, mã phù hợp là 72112320.'},
  {'prompt': 'Tôi nhập quả hồng xiêm tươi từ Thái Lan thì dùng mã HS nào?',
   'response': 'Bạn dùng mã 08109093 nhé, mã này dành cho quả hồng xiêm (sapôchê) tươi.'},
  {'prompt': 'Khăn tay kh

In [14]:
from transformers import AutoTokenizer

model_name = "VietAI/vit5-base"  # hoặc mô hình khác như "t5-base", "gpt2", v.v.
tokenizer = AutoTokenizer.from_pretrained("VietAI/vit5-base")  

print("Max token length:", tokenizer.model_max_length)


Max token length: 1000000000000000019884624838656
